In [ ]:
# 2. Pré-processamento, Modelagem e Avaliação

Neste notebook, vamos preparar os dados para serem usados em modelos de Machine Learning. As etapas incluem:
1.  Limpeza e tratamento de dados.
2.  Conversão de variáveis categóricas em numéricas.
3.  Escalonamento de features.
4.  Divisão em conjuntos de treino e teste.
5.  Treinamento e avaliação de 3 modelos de classificação.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

In [ ]:
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

## 1. Pré-processamento dos Dados

In [ ]:
# A coluna TotalCharges tem alguns valores em branco que precisam ser tratados.
# Convertemos para numérico, forçando os erros (espaços em branco) para NaN (Not a Number)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Verificando os NaNs criados
print(f"Valores nulos em TotalCharges após conversão: {df['TotalCharges'].isnull().sum()}")

# Preenchendo os valores nulos com a mediana da coluna
median_total_charges = df['TotalCharges'].median()
df['TotalCharges'].fillna(median_total_charges, inplace=True)

print(f"Valores nulos em TotalCharges após preenchimento: {df['TotalCharges'].isnull().sum()}")

# Removendo a coluna de ID, que não é útil para o modelo
df.drop('customerID', axis=1, inplace=True)

In [ ]:
# Identificando colunas categóricas e numéricas
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
# A variável alvo 'Churn' está aqui, vamos tratá-la separadamente
categorical_cols.remove('Churn')

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

# Usando One-Hot Encoding para as features categóricas
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Usando Label Encoding para a variável alvo (Churn)
le = LabelEncoder()
df['Churn'] = le.fit_transform(df['Churn']) # No -> 0, Yes -> 1

df.head()

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

# Escalonando as features numéricas
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

X.head()


In [ ]:
# Usamos stratify=y para garantir que a proporção de Churn seja a mesma nos conjuntos de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Formato de X_train: {X_train.shape}")
print(f"Formato de X_test: {X_test.shape}")

## 2. Treinamento e Avaliação dos Modelos

In [ ]:
models = {
    'Regressão Logística': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}

for name, model in models.items():
    # Treinamento
    model.fit(X_train, y_train)
    
    # Previsão
    y_pred = model.predict(X_test)
    
    # Avaliação
    print(f"--- {name} ---")
    print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}")
    print(f"AUC: {roc_auc_score(y_test, y_pred):.4f}")
    print("\nRelatório de Classificação:")
    print(classification_report(y_test, y_pred))
    print("-" * 40 + "\n")

## 3. Análise do Melhor Modelo (XGBoost)

Com base nas métricas, especialmente no equilíbrio entre precisão e recall, o XGBoost se mostrou um forte candidato. Vamos visualizar sua performance com uma Matriz de Confusão.

In [ ]:
# Usando o modelo XGBoost já treinado
xgb_model = models['XGBoost']
y_pred_xgb = xgb_model.predict(X_test)

# Gerando a matriz de confusão
cm = confusion_matrix(y_test, y_pred_xgb, labels=xgb_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=xgb_model.classes_)

# Plotando a matriz
disp.plot()
plt.title('Matriz de Confusão - XGBoost')

# Salvando a imagem para o README
plt.savefig('../images/confusion_matrix.png')

plt.show()